# COM299 Applied Data Science
## Validate STATS19 Table Relationships

This notebook validates the relationships between the 2025 STATS19 casualty, vehicle, and collision datasets.

The purpose is to confirm that the tables can be joined reliably, identify any unmatched records, and formally justify the unit of analysis before building the analytical dataset.

### Load the three source datasets

All three raw STATS19 tables are loaded so their key relationships can be tested directly.

The raw files remain unchanged, and this notebook focuses only on validating how the tables connect.

In [1]:
import pandas as pd

casualties = pd.read_csv("../data/raw/dft-road-casualty-statistics-casualty-2025.csv")
vehicles = pd.read_csv("../data/raw/dft-road-casualty-statistics-vehicle-2025.csv")
collisions = pd.read_csv("../data/raw/dft-road-casualty-statistics-collision-2025.csv")

C:\Users\Richa\AppData\Local\Temp\ipykernel_13068\1303496463.py:4: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicles = pd.read_csv("../data/raw/dft-road-casualty-statistics-vehicle-2025.csv")


### Check the size of each table

The number of rows in each dataset is checked so the table relationships can be interpreted in context before any joins are attempted.

In [2]:
print("Casualties:", casualties.shape)
print("Vehicles:", vehicles.shape)
print("Collisions:", collisions.shape)

Casualties: (127883, 23)
Vehicles: (183948, 32)
Collisions: (101525, 44)


**Result:** The three STATS19 tables contain different numbers of records:

- Casualties: 127,883 rows
- Vehicles: 183,948 rows
- Collisions: 101,525 rows

The different row counts are expected because the tables represent different units.

A single collision can involve multiple vehicles and multiple casualties, so the collision table has fewer rows than the vehicle and casualty tables.

The next step is to test these relationships directly rather than assuming that every record links successfully across the datasets.


### Confirm collision key uniqueness

The collision table should contain one record per `collision_index`.

This must be confirmed before testing a many-to-one join from casualties to collisions, because duplicate collision keys could incorrectly duplicate casualty records during a merge.

In [3]:
collisions["collision_index"].duplicated().sum()

np.int64(0)

### Validate casualty-to-collision matching

Each casualty record should link to one collision through `collision_index`.

A left merge with an indicator is used to check whether every casualty finds a matching collision record while preserving the full casualty table.

In [4]:
casualty_collision_check = casualties.merge(
    collisions[["collision_index"]],
    on="collision_index",
    how="left",
    indicator=True,
    validate="many_to_one"
)

casualty_collision_check["_merge"].value_counts()

_merge
both          127883
left_only          0
right_only         0
Name: count, dtype: int64

**Result:** All 127,883 casualty records successfully matched to a collision record.

- `both`: 127,883
- `left_only`: 0
- `right_only`: 0

This confirms that every casualty in the 2025 casualty table can be linked to exactly one collision using `collision_index`.

The `validate="many_to_one"` check also passed, supporting the expected relationship of many casualties to one collision.


### Confirm vehicle key uniqueness

Vehicle records are identified by the combination of `collision_index` and `vehicle_reference`.

Before joining casualties to vehicles, this composite key should be checked for duplicates to confirm that each vehicle is unique within its collision.

In [6]:
vehicles.duplicated(
    subset=["collision_index", "vehicle_reference"]
).sum()

np.int64(0)

**Result:** No duplicate combinations of `collision_index` and `vehicle_reference` were found in the vehicle table.

This confirms that the composite key uniquely identifies each vehicle within its collision.

The vehicle table is therefore suitable for a many-to-one join from casualty records using `collision_index` and `vehicle_reference`.

### Validate casualty-to-vehicle matching

Each casualty record should link to the vehicle associated with that casualty using `collision_index` and `vehicle_reference`.

A left merge with an indicator is used to check whether every casualty finds a matching vehicle record while preserving the full casualty table.

In [7]:
casualty_vehicle_check = casualties.merge(
    vehicles[["collision_index", "vehicle_reference"]],
    on=["collision_index", "vehicle_reference"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

casualty_vehicle_check["_merge"].value_counts()

_merge
both          127883
left_only          0
right_only         0
Name: count, dtype: int64

**Result:** All 127,883 casualty records successfully matched to a vehicle record.

- `both`: 127,883
- `left_only`: 0
- `right_only`: 0

This confirms that every casualty in the 2025 casualty table can be linked to exactly one vehicle using the combination of `collision_index` and `vehicle_reference`.

The `validate="many_to_one"` check also passed, confirming the expected relationship of many casualty records to one vehicle record.

### Check that the join preserves casualty row count

A valid many-to-one join should add vehicle information without increasing or reducing the number of casualty records.

The row count before and after the merge is therefore compared.

In [8]:
print("Original casualty rows:", len(casualties))
print("After casualty-vehicle join:", len(casualty_vehicle_check))

Original casualty rows: 127883
After casualty-vehicle join: 127883


**Result:** The casualty-to-vehicle merge preserved the original casualty row count.

- Original casualty rows: 127,883
- After join: 127,883

This confirms that the join added vehicle linkage without duplicating or dropping casualty records.

Together with the successful match rate and `many_to_one` validation, this provides strong evidence that `collision_index` plus `vehicle_reference` is a reliable key for linking casualties to vehicles.

### Check that the casualty-to-collision join preserves row count

The casualty-to-collision relationship has already matched successfully.

The row count is checked again to confirm that adding collision-level information does not duplicate or remove casualty records.

In [9]:
print("Original casualty rows:", len(casualties))
print("After casualty-collision join:", len(casualty_collision_check))

Original casualty rows: 127883
After casualty-collision join: 127883


**Result:** The casualty-to-collision merge preserved the original casualty row count.

- Original casualty rows: 127,883
- After join: 127,883

This confirms that collision-level information can be added without duplicating or removing casualty records.

Together with the successful match rate and `many_to_one` validation, this supports using the casualty table as the base analytical dataset.